In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

df = pd.read_csv('data/sales.csv')
df['date'] = pd.to_datetime(df['date'])
df['revenue'] = df['qty'] * df['price']

df['week'] = df['date'].dt.to_period('W-MON').dt.start_time

weekly_sales = df.groupby(['week', 'category'])['revenue'].sum().reset_index()
print(f"Данные с {weekly_sales['week'].min().date()} по {weekly_sales['week'].max().date()}")
print(f"Всего недель: {weekly_sales['week'].nunique()}")

Данные с 2020-12-29 по 2023-12-26
Всего недель: 157


In [2]:

max_date = weekly_sales['week'].max()
test_start_date = max_date - pd.Timedelta(weeks=7)

print(f"Тест начинается с: {test_start_date.date()}")

train = weekly_sales[weekly_sales['week'] < test_start_date].copy()
test = weekly_sales[weekly_sales['week'] >= test_start_date].copy()

print(f"Обучающая выборка: {len(train)} строк, Тестовая: {len(test)} строк")

Тест начинается с: 2023-11-07
Обучающая выборка: 447 строк, Тестовая: 24 строк


In [3]:
categories = weekly_sales['category'].unique()
forecast_results = {}

for cat in categories:
    cat_data = weekly_sales[weekly_sales['category'] == cat].sort_values('week').reset_index(drop=True)
    
    train_cat = cat_data[cat_data['week'] < test_start_date].copy()
    test_cat = cat_data[cat_data['week'] >= test_start_date].copy()
    
    last_4_weeks_avg = train_cat['revenue'].tail(4).mean()
    forecast_ma = [last_4_weeks_avg] * len(test_cat)
    
    x_train = np.arange(len(train_cat))
    y_train = train_cat['revenue'].values
    coeffs = np.polyfit(x_train, y_train, deg=1)
    x_test = np.arange(len(train_cat), len(train_cat) + len(test_cat))
    forecast_trend = np.polyval(coeffs, x_test)
    
    mae_ma = mean_absolute_error(test_cat['revenue'], forecast_ma)
    mae_trend = mean_absolute_error(test_cat['revenue'], forecast_trend)
    
    forecast_results[cat] = {
        'test_dates': test_cat['week'].values,
        'actual': test_cat['revenue'].values,
        'forecast_ma': forecast_ma,
        'forecast_trend': forecast_trend,
        'mae_ma': mae_ma,
        'mae_trend': mae_trend
    }
    
    print(f"{cat}")
    print(f"MAE (Скользящее среднее): {mae_ma:.2f}")
    print(f"MAE (Линейный тренд): {mae_trend:.2f}\n")

Одежда
MAE (Скользящее среднее): 177059.91
MAE (Линейный тренд): 243564.73

Продукты
MAE (Скользящее среднее): 35314.77
MAE (Линейный тренд): 37868.76

Электроника
MAE (Скользящее среднее): 2544661.02
MAE (Линейный тренд): 2279578.42



In [4]:
fig, axes = plt.subplots(len(categories), 1, figsize=(14, 6 * len(categories)))
if len(categories) == 1:
    axes = [axes]

for i, cat in enumerate(categories):
    ax = axes[i]
    res = forecast_results[cat]
    
    ax.plot(res['test_dates'], res['actual'], marker='o', label='Факт', color='black', linewidth=2)
    ax.plot(res['test_dates'], res['forecast_ma'], marker='s', linestyle='--', label=f'Скользящее среднее (MAE={res["mae_ma"]:.0f})', color='blue')
    ax.plot(res['test_dates'], res['forecast_trend'], marker='^', linestyle=':', label=f'Линейный тренд (MAE={res["mae_trend"]:.0f})', color='red')
    
    ax.set_title(f'Прогноз продаж: {cat}')
    ax.set_xlabel('Дата')
    ax.set_ylabel('Выручка')
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

<Figure size 1400x1800 with 3 Axes>